# ✈️ Flight Ticket Price Prediction
### Applying, Evaluating & Comparing 5 Regression Algorithms with 10-Fold Cross-Validation

| | |
|---|---|
| 🔗 **Live Web App (Deployment)** | **[https://flight-prediction-model-24jznugnvxzgr5mksxuxwf.streamlit.app](https://flight-prediction-model-24jznugnvxzgr5mksxuxwf.streamlit.app/)** |
| 💻 **GitHub Repository** | [https://github.com/rameenejaz/flight-prediction-model](https://github.com/rameenejaz/flight-prediction-model.git) |
| 👤 **Student Name** | Rameen Muzammil Ejaz |
| 🆔 **Registration No.** | FA24-BSE-160 |
| 📚 **Course** | Machine Learning Fundamentals (AIC354) |
| 👨‍🏫 **Instructor** | Dr. Rao Muhammad Adeel Nawab |
| 📅 **Assignment** | Assignment 3, Task 2 (Fall 2026) |



## 📑 Table of Contents
1. Problem Statement
2. Dataset Description
3. Algorithms Selected
4. Setup: Import Libraries
5. Load the Full Dataset
6. Understand the Data
7. Exploratory Data Analysis (charts)
8. Preprocessing
9. 10-Fold Cross-Validation of 5 Algorithms (with progress bar)
10. Results Tables
11. Comparison Charts
12. Best Model Analysis
13. Train Final Model & Save for Deployment
14. Deployment (Streamlit Community Cloud)
15. Conclusion

## 1. Problem Statement

Airline ticket prices change constantly depending on the airline, route, time of day, number of stops,
travel class and how far in advance the ticket is booked. Travellers find it hard to know whether a fare is reasonable.

**Goal:** given the details of a flight, predict its **ticket price (in Indian Rupees, ₹)**.

| Item | Description |
|---|---|
| **ML task** | Supervised learning, **Regression** (the target is a continuous number) |
| **Input (features)** | airline, source city, departure time, stops, arrival time, destination city, class, duration, days left |
| **Output (target)** | `price` (₹) |
| **Evaluation** | 10-Fold Cross-Validation on the **full dataset** |
| **Main metric** | **RMSE** (Root Mean Squared Error), same as the Car Price Prediction System. Lower is better |
| **Extra metrics** | MAE, R², training time and prediction time |

This is the same type of problem as the **Car Price Prediction System**: tabular data, mixed categorical and numeric
features, and a price as the target.

## 2. Dataset Description

**Source:** Kaggle, *Flight Price Prediction* by Shubham Bathwal
(https://www.kaggle.com/datasets/shubhambathwal/flight-price-prediction). File used: **`Clean_Dataset.csv`**.

The data was collected from the travel booking website *EaseMyTrip* and contains booking options for flights between
India's six biggest metro cities. We use the **FULL dataset (~300,000 rows)**; the exact size is printed in Section 5.

| Column | Type | Meaning |
|---|---|---|
| `airline` | categorical | Airline company (6 airlines) |
| `flight` | categorical | Flight code, an identifier (**dropped**, see Section 8) |
| `source_city` | categorical | City the flight departs from |
| `departure_time` | categorical | Time-of-day bin of departure (e.g. Morning, Night) |
| `stops` | categorical | Number of stops: zero / one / two_or_more |
| `arrival_time` | categorical | Time-of-day bin of arrival |
| `destination_city` | categorical | City the flight lands in |
| `class` | categorical | Economy or Business |
| `duration` | numeric | Total travel time in hours |
| `days_left` | numeric | Days between booking date and travel date |
| **`price`** | numeric | **Target**: ticket price in ₹ |

## 3. Algorithms Selected

The assignment requires 5 algorithms that were **not** used in the Car Price Prediction System.
I chose one algorithm from each of several different families so the comparison is meaningful:

| # | Algorithm | Family | Key idea |
|---|---|---|---|
| 1 | **Ridge Regression** | Linear (regularised) | Linear regression with an L2 penalty that shrinks coefficients to reduce overfitting. Fast, simple baseline. |
| 2 | **Extra Trees Regressor** | Bagging ensemble | Many decision trees with *random* split thresholds; predictions are averaged. Lower variance than a single tree. |
| 3 | **AdaBoost Regressor** | Boosting (classic) | Trees are trained one after another; each new tree focuses more on the samples the previous trees got wrong. |
| 4 | **XGBoost Regressor** | Gradient boosting | Each new tree fits the remaining errors (gradients) of the ensemble; built-in regularisation, very optimised. |
| 5 | **LightGBM Regressor** | Gradient boosting (histogram, leaf-wise) | Like XGBoost but grows trees leaf-by-leaf and bins features into histograms, which makes it very fast on large data. |

None of these five algorithms is used in the Car Price Prediction System, as the assignment requires.

## 4. Setup: Import Libraries

In [ ]:
# Fixed library versions. The SAME versions are listed in the web app's requirements.txt, so the model
# trained here loads identically on Streamlit Cloud (they install on Python 3.12, 3.13 and 3.14).
import importlib.metadata as md, subprocess, sys, os

PINNED = {'scikit-learn': '1.8.0', 'numpy': '2.4.4', 'pandas': '3.0.2', 'joblib': '1.5.3',
          'xgboost': '3.4.1', 'lightgbm': '4.7.0', 'tqdm': None, 'seaborn': None}

def current(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

todo = [f'{p}=={v}' if v else p for p, v in PINNED.items()
        if (v and current(p) != v) or (not v and current(p) is None)]
if todo:
    print('Installing:', ', '.join(todo))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *todo], check=True)
    print('\n✅ Installed. The runtime restarts now. When it reconnects, click  Runtime ▸ Run all  again.')
    os.kill(os.getpid(), 9)          # restart so the new versions are loaded
else:
    print('✅ All library versions are correct:', ', '.join(f'{p} {current(p)}' for p in PINNED))

In [ ]:
import os, time, json, platform, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from tqdm.auto import tqdm
import joblib

import sklearn
from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
import xgboost
from xgboost import XGBRegressor
import lightgbm
from lightgbm import LGBMRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'axes.titleweight': 'bold', 'axes.titlesize': 13})

# One fixed colour per model, used in every chart
MODEL_COLORS = {
    'Ridge Regression': '#8C96A3',
    'Extra Trees':      '#2E8B57',
    'AdaBoost':         '#E0A100',
    'XGBoost':          '#C8553D',
    'LightGBM':         '#3366CC',
}

print(f'Python       : {platform.python_version()}')
for lib in [np, pd, sklearn, xgboost, lightgbm, joblib]:
    print(f'{lib.__name__:<13}: {lib.__version__}')
print(f'CPU cores    : {os.cpu_count()}')

In [ ]:
# ───── Configuration ─────
DATA_FILE    = 'Clean_Dataset.csv'
TARGET       = 'price'
N_FOLDS      = 10       # assignment requirement: 10-fold cross-validation
RANDOM_STATE = 42       # makes results reproducible
DEBUG_SAMPLE = None     # None = FULL dataset (required). Set e.g. 2000 only for a quick test run.
MAX_MODEL_MB = 95       # GitHub rejects files > 100 MB, so the deployed model must stay under this

## 5. Load the Full Dataset
If `Clean_Dataset.csv` is not already in the Colab session, an **upload button** appears. Choose the file you downloaded from Kaggle.

In [ ]:
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print(f'Please upload {DATA_FILE} (downloaded from Kaggle)...')
        uploaded = files.upload()
        DATA_FILE = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f'{DATA_FILE} not found. Put it in the same folder as this notebook.')

df_raw = pd.read_csv(DATA_FILE)
if DEBUG_SAMPLE:
    df_raw = df_raw.sample(DEBUG_SAMPLE, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f'⚠️  DEBUG MODE: using only {DEBUG_SAMPLE:,} rows. Set DEBUG_SAMPLE = None for the real run!')

print(f'Loaded {DATA_FILE}: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head()

## 6. Understand the Data

In [ ]:
overview = pd.DataFrame({
    'dtype': df_raw.dtypes.astype(str),
    'missing': df_raw.isna().sum(),
    'unique values': df_raw.nunique(),
    'example': df_raw.iloc[0],
})
print(f'Duplicate rows: {df_raw.duplicated().sum():,}')
overview

In [ ]:
df_raw.describe().T

In [ ]:
# Unique values of each categorical column
cat_cols_raw = ['airline', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'class']
for c in cat_cols_raw:
    counts = df_raw[c].value_counts()
    print(f'{c:<17} ({len(counts)}): ' + ', '.join(f'{k} [{v:,}]' for k, v in counts.items()))

## 7. Exploratory Data Analysis
These charts show how each feature relates to the price, before any modelling.

In [ ]:
rupee = mticker.FuncFormatter(lambda v, _: f'₹{v/1000:,.0f}k')

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# (a) target distribution
sns.histplot(df_raw[TARGET], bins=80, ax=axes[0, 0], color='#3366CC')
axes[0, 0].set(title='Distribution of Ticket Price', xlabel='Price', ylabel='Number of tickets')
axes[0, 0].xaxis.set_major_formatter(rupee)

# (b) price by class
sns.boxplot(data=df_raw, x='class', y=TARGET, ax=axes[0, 1], palette=['#3366CC', '#C8553D'], showfliers=False)
axes[0, 1].set(title='Price by Travel Class', xlabel='', ylabel='Price')
axes[0, 1].yaxis.set_major_formatter(rupee)

# (c) price by airline and class
order = df_raw.groupby('airline')[TARGET].median().sort_values().index
sns.barplot(data=df_raw, x='airline', y=TARGET, hue='class', order=order, ax=axes[1, 0],
            palette=['#3366CC', '#C8553D'], errorbar=None)
axes[1, 0].set(title='Average Price by Airline', xlabel='', ylabel='Average price')
axes[1, 0].yaxis.set_major_formatter(rupee)

# (d) price by number of stops
sns.barplot(data=df_raw, x='stops', y=TARGET, hue='class', order=['zero', 'one', 'two_or_more'],
            ax=axes[1, 1], palette=['#3366CC', '#C8553D'], errorbar=None)
axes[1, 1].set(title='Average Price by Number of Stops', xlabel='Stops', ylabel='Average price')
axes[1, 1].yaxis.set_major_formatter(rupee)

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# (e) how booking early affects price
trend = df_raw.groupby(['days_left', 'class'])[TARGET].mean().reset_index()
sns.lineplot(data=trend, x='days_left', y=TARGET, hue='class', ax=axes[0], marker='o', markersize=4,
             palette=['#C8553D', '#3366CC'])
axes[0].set(title='Average Price vs. Days Left Before Departure', xlabel='Days left', ylabel='Average price')
axes[0].yaxis.set_major_formatter(rupee)

# (f) route heat-map (economy only, so business fares don't dominate)
route = (df_raw[df_raw['class'] == 'Economy']
         .pivot_table(index='source_city', columns='destination_city', values=TARGET, aggfunc='mean'))
sns.heatmap(route, annot=True, fmt=',.0f', cmap='YlOrRd', ax=axes[1], cbar_kws={'label': 'Avg economy price (₹)'})
axes[1].set(title='Average Economy Price by Route', xlabel='Destination', ylabel='Source')

plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sample = df_raw.sample(min(20000, len(df_raw)), random_state=RANDOM_STATE)
sns.scatterplot(data=sample, x='duration', y=TARGET, hue='class', alpha=0.25, s=10, ax=ax,
                palette=['#3366CC', '#C8553D'])
ax.set(title='Price vs. Flight Duration (20k-row sample)', xlabel='Duration (hours)', ylabel='Price')
ax.yaxis.set_major_formatter(rupee)
plt.tight_layout(); plt.show()

**EDA observations:**
- **Class** is by far the strongest driver: business fares are many times higher than economy, which makes the price distribution bimodal.
- Prices rise sharply when **days_left** is small (last-minute bookings are expensive).
- Prices differ between **airlines**, **routes** and **number of stops**.
- The relationships are non-linear, so tree-based ensembles are expected to beat the linear model.

## 8. Preprocessing

| Step | What & why |
|---|---|
| Drop `Unnamed: 0` | It is only the row index saved in the CSV, not a feature. |
| Drop `flight` | It is a flight-code **identifier** with ~1,500 values. A user of the web app would not know it, and it mostly lets the model memorise individual flights instead of learning general patterns. |
| `stops`, `class` → **Ordinal encoding** | These have a natural order (zero < one < two_or_more; Economy < Business). |
| Other categoricals → **One-Hot encoding** | airline, cities and time bins have no order, so each value becomes its own 0/1 column. `handle_unknown='ignore'` keeps the app from crashing on unexpected input. |
| `duration`, `days_left` → **Standard scaling** | Needed for Ridge (regularisation is scale-sensitive); harmless for tree models. |

All preprocessing lives **inside a scikit-learn `Pipeline`**, which gives two benefits:
1. During cross-validation the encoders/scaler are fitted **only on the training folds**, so there is **no data leakage**.
2. The saved model file contains preprocessing + model together, so the web app can pass in raw values directly.

In [ ]:
df = df_raw.drop(columns=[c for c in ['Unnamed: 0', 'flight'] if c in df_raw.columns])

ORDINAL_COLS = ['stops', 'class']
ORDINAL_CATS = [['zero', 'one', 'two_or_more'], ['Economy', 'Business']]
NOMINAL_COLS = ['airline', 'source_city', 'departure_time', 'arrival_time', 'destination_city']
NUMERIC_COLS = ['duration', 'days_left']
FEATURES     = ORDINAL_COLS + NOMINAL_COLS + NUMERIC_COLS

X = df[FEATURES]
y = df[TARGET].astype(float)

preprocessor = ColumnTransformer([
    ('ord', OrdinalEncoder(categories=ORDINAL_CATS), ORDINAL_COLS),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), NOMINAL_COLS),
    ('num', StandardScaler(), NUMERIC_COLS),
])

n_out = preprocessor.fit(X.head(5000)).transform(X.head(1)).shape[1]
print(f'X: {X.shape[0]:,} rows × {X.shape[1]} raw features  →  {n_out} model-ready columns after encoding')
print(f'y: price from ₹{y.min():,.0f} to ₹{y.max():,.0f} (mean ₹{y.mean():,.0f})')

## 9. 10-Fold Cross-Validation of 5 Algorithms

**How K-Fold CV works:** the full dataset is shuffled and split into 10 equal parts (folds). Each model is trained 10 times;
each time one fold is held out for testing and the other 9 are used for training. Every row is used for testing exactly once,
and we report the **mean ± standard deviation** of the 10 test scores. All 5 models use **exactly the same 10 splits**,
so the comparison is fair.

**Hyper-parameters:** reasonable settings chosen to balance accuracy and Colab run time (no tuning was done,
so every model is compared on equal footing).

In [ ]:
def make_models():
    return {
        'Ridge Regression': Ridge(alpha=1.0),
        'Extra Trees': ExtraTreesRegressor(
            n_estimators=60, min_samples_leaf=3, max_features=0.7,
            n_jobs=-1, random_state=RANDOM_STATE),
        'AdaBoost': AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE),
            n_estimators=30, learning_rate=0.5, random_state=RANDOM_STATE),
        'XGBoost': XGBRegressor(
            n_estimators=400, learning_rate=0.1, max_depth=8, subsample=0.9,
            tree_method='hist', n_jobs=-1, random_state=RANDOM_STATE),
        'LightGBM': LGBMRegressor(
            n_estimators=400, learning_rate=0.1, num_leaves=127,
            n_jobs=-1, random_state=RANDOM_STATE, verbose=-1),
    }

def make_pipeline(model):
    return Pipeline([('prep', clone(preprocessor)), ('model', clone(model))])

models = make_models()
pd.DataFrame({'Model': list(models), 'Estimator': [type(m).__name__ for m in models.values()]})

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
folds = list(kf.split(X))                       # identical splits for every model

fold_records = []
oof_pred = {name: np.zeros(len(y)) for name in models}   # out-of-fold predictions (used in Section 12)

start_all = time.perf_counter()
with tqdm(total=len(models) * N_FOLDS, desc='10-Fold CV', unit='fit',
          bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} fits [elapsed {elapsed} | remaining {remaining}] {postfix}') as pbar:
    for name, model in models.items():
        for k, (tr_idx, te_idx) in enumerate(folds, start=1):
            pbar.set_postfix_str(f'{name}, fold {k}/{N_FOLDS}')
            pipe = make_pipeline(model)

            t0 = time.perf_counter()
            pipe.fit(X.iloc[tr_idx], y.iloc[tr_idx])
            train_s = time.perf_counter() - t0

            t0 = time.perf_counter()
            pred = pipe.predict(X.iloc[te_idx])
            pred_s = time.perf_counter() - t0

            y_true = y.iloc[te_idx]
            oof_pred[name][te_idx] = pred
            fold_records.append({
                'Model': name, 'Fold': k,
                'RMSE': np.sqrt(mean_squared_error(y_true, pred)),
                'MAE':  mean_absolute_error(y_true, pred),
                'R2':   r2_score(y_true, pred),
                'Train time (s)':   train_s,
                'Predict time (s)': pred_s,
            })
            pbar.update(1)
        last = [r for r in fold_records if r['Model'] == name]
        tqdm.write(f'✔ {name:<17} mean RMSE = ₹{np.mean([r["RMSE"] for r in last]):,.0f}   '
                   f'total train time = {sum(r["Train time (s)"] for r in last):,.1f}s')

total_minutes = (time.perf_counter() - start_all) / 60
fold_df = pd.DataFrame(fold_records)
print(f'\nFinished {len(fold_df)} fits in {total_minutes:.1f} minutes.')

## 10. Results Tables

### 10.1 RMSE of every model on every fold (₹)

In [ ]:
rmse_table = fold_df.pivot(index='Fold', columns='Model', values='RMSE')[list(models)]
rmse_table.loc['Mean'] = rmse_table.mean()
rmse_table.loc['Std']  = rmse_table.iloc[:N_FOLDS].std()
(rmse_table.style
     .format('{:,.0f}')
     .background_gradient(cmap='RdYlGn_r', axis=1, subset=pd.IndexSlice[1:N_FOLDS, :])
     .set_properties(subset=pd.IndexSlice[['Mean', 'Std'], :], **{'font-weight': 'bold'})
     .set_caption('Per-fold RMSE (green = better within each fold)'))

### 10.2 Summary comparison (sorted by mean RMSE)

In [ ]:
summary = (fold_df.groupby('Model')
           .agg(RMSE_mean=('RMSE', 'mean'), RMSE_std=('RMSE', 'std'),
                MAE_mean=('MAE', 'mean'), R2_mean=('R2', 'mean'), R2_std=('R2', 'std'),
                Train_total_s=('Train time (s)', 'sum'), Train_per_fold_s=('Train time (s)', 'mean'),
                Predict_per_fold_s=('Predict time (s)', 'mean'))
           .sort_values('RMSE_mean'))
summary.insert(0, 'Rank', range(1, len(summary) + 1))
summary['Speed vs fastest'] = summary['Train_total_s'] / summary['Train_total_s'].min()

display_summary = pd.DataFrame({
    'Rank': summary['Rank'],
    'RMSE (₹)': [f'{m:,.0f} ± {s:,.0f}' for m, s in zip(summary.RMSE_mean, summary.RMSE_std)],
    'MAE (₹)': summary['MAE_mean'].map('{:,.0f}'.format),
    'R²': [f'{m:.4f} ± {s:.4f}' for m, s in zip(summary.R2_mean, summary.R2_std)],
    'Total train time (10 folds)': summary['Train_total_s'].map(lambda s: f'{s:,.1f} s'),
    'Avg train time / fold': summary['Train_per_fold_s'].map(lambda s: f'{s:,.2f} s'),
    'Avg predict time / fold': summary['Predict_per_fold_s'].map(lambda s: f'{s:,.3f} s'),
    'Slower than fastest': summary['Speed vs fastest'].map(lambda v: f'{v:,.1f}×'),
})
best_name = summary.index[0]
(display_summary.style
     .apply(lambda r: ['background-color:#d9f2e3; font-weight:bold' if r.name == best_name else '' for _ in r], axis=1)
     .set_caption(f'🏆 Best model by mean RMSE: {best_name}'))

### 10.3 Timing comparison (head-to-head)

In [ ]:
# How many times slower is the row model than the column model (training time over all 10 folds)
t = summary['Train_total_s']
speed_matrix = pd.DataFrame(t.values[:, None] / t.values[None, :], index=t.index, columns=t.index)
(speed_matrix.style.format('{:,.1f}×').background_gradient(cmap='Blues')
     .set_caption('Training time ratio: row model ÷ column model'))

## 11. Comparison Charts

In [ ]:
order  = list(summary.index)                       # best → worst
colors = [MODEL_COLORS[m] for m in order]

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# (1) mean RMSE with std error bars
ax = axes[0]
bars = ax.barh(order[::-1], summary['RMSE_mean'][::-1], xerr=summary['RMSE_std'][::-1],
               color=colors[::-1], capsize=5, edgecolor='white')
for b, v, sd in zip(bars, summary['RMSE_mean'][::-1], summary['RMSE_std'][::-1]):
    ax.text(v + sd, b.get_y() + b.get_height() / 2, f'  ₹{v:,.0f}', va='center', fontweight='bold')
ax.set(title='Mean RMSE ± std across 10 folds (lower is better)', xlabel='RMSE (₹)')
ax.set_xlim(0, (summary['RMSE_mean'] + summary['RMSE_std']).max() * 1.3)

# (2) distribution of RMSE over the 10 folds
ax = axes[1]
sns.boxplot(data=fold_df, y='Model', x='RMSE', order=order, palette=MODEL_COLORS, ax=ax)
sns.stripplot(data=fold_df, y='Model', x='RMSE', order=order, color='black', size=4, alpha=.6, ax=ax)
ax.set_xscale('log')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.xaxis.set_minor_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}' if str(int(v))[0] in '25' else ''))
ax.set(title='Spread of RMSE over the 10 folds (log scale)', xlabel='RMSE (₹)', ylabel='')

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# (3) R²
ax = axes[0]
ax.bar(order, summary['R2_mean'], yerr=summary['R2_std'], color=colors, capsize=5)
for i, v in enumerate(summary['R2_mean']):
    ax.text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold')
ax.set(title='Mean R² (higher is better)', ylabel='R²', ylim=(max(0, summary['R2_mean'].min() - 0.05), 1.01))
ax.tick_params(axis='x', rotation=15)

# (4) MAE
ax = axes[1]
ax.bar(order, summary['MAE_mean'], color=colors)
for i, v in enumerate(summary['MAE_mean']):
    ax.text(i, v * 1.02, f'₹{v:,.0f}', ha='center', fontweight='bold')
ax.set(title='Mean MAE (lower is better)', ylabel='MAE (₹)')
ax.tick_params(axis='x', rotation=15)

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# (5) training time (log scale because Ridge is orders of magnitude faster)
ax = axes[0]
by_time = summary.sort_values('Train_total_s')
ax.barh(by_time.index, by_time['Train_total_s'], color=[MODEL_COLORS[m] for m in by_time.index])
for i, v in enumerate(by_time['Train_total_s']):
    ax.text(v * 1.1, i, f'{v:,.1f} s', va='center', fontweight='bold')
ax.set_xscale('log')
ax.set(title='Total training time for 10 folds (log scale)', xlabel='Seconds')
ax.set_xlim(by_time['Train_total_s'].min() / 2, by_time['Train_total_s'].max() * 6)

# (6) prediction time
ax = axes[1]
by_pred = summary.sort_values('Predict_per_fold_s')
ax.barh(by_pred.index, by_pred['Predict_per_fold_s'], color=[MODEL_COLORS[m] for m in by_pred.index])
for i, v in enumerate(by_pred['Predict_per_fold_s']):
    ax.text(v * 1.02, i, f'{v:.3f} s', va='center', fontweight='bold')
ax.set(title=f'Average time to predict one test fold (~{len(y)//N_FOLDS:,} rows)', xlabel='Seconds')
ax.set_xlim(0, by_pred['Predict_per_fold_s'].max() * 1.3)

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# (7) accuracy vs. speed trade-off
ax = axes[0]
for m in order:
    ax.scatter(summary.loc[m, 'Train_total_s'], summary.loc[m, 'RMSE_mean'], s=300,
               color=MODEL_COLORS[m], edgecolor='black', zorder=3, label=f"{m} (₹{summary.loc[m, 'RMSE_mean']:,.0f})")
ax.set_xscale('log'); ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.yaxis.set_minor_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.legend(fontsize=9, loc='best')
ax.set(title='Accuracy vs. Speed (bottom-left is best)', xlabel='Total training time, seconds (log)', ylabel='Mean RMSE, ₹ (log)')

# (8) RMSE fold by fold. A model far worse than the rest would flatten the others, so it is left out of this zoomed view
ax = axes[1]
best_rmse = summary['RMSE_mean'].min()
shown = [m for m in order if summary.loc[m, 'RMSE_mean'] < 2 * best_rmse]
hidden = [m for m in order if m not in shown]
for m in shown:
    d = fold_df[fold_df.Model == m]
    ax.plot(d['Fold'], d['RMSE'], marker='o', label=m, color=MODEL_COLORS[m], linewidth=2)
title = 'RMSE on each fold' + (f"\n(zoomed: {', '.join(hidden)} not shown, see boxplot)" if hidden else '')
ax.set(title=title, xlabel='Fold', ylabel='RMSE (₹)', xticks=range(1, N_FOLDS + 1))
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))

plt.tight_layout(); plt.show()

### 11.1 Is the best model significantly better than the runner-up?

In [ ]:
# Paired t-test on the 10 fold scores (same folds for both models → paired test).
# Note: CV folds share training data, so this test is approximate; treat it as supporting evidence.
first, second = order[0], order[1]
a = fold_df[fold_df.Model == first].sort_values('Fold')['RMSE'].values
b = fold_df[fold_df.Model == second].sort_values('Fold')['RMSE'].values
t_stat, p_val = stats.ttest_rel(a, b)
wins = int((a < b).sum())
print(f'{first} vs {second}')
print(f'  mean RMSE difference : ₹{(b - a).mean():,.0f} in favour of {first}')
print(f'  {first} wins on       : {wins}/{N_FOLDS} folds')
print(f'  paired t-test p-value : {p_val:.4g} → ' + ('significant at 5% level' if p_val < 0.05 else 'NOT significant at 5% level'))

## 12. Best Model Analysis
The plots below use **out-of-fold predictions**: every price was predicted by a model that had **not** seen that row during training,
so they show honest, unseen-data performance on all ~300k rows.

In [ ]:
pred_best = oof_pred[best_name]
resid = y.values - pred_best

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
hb = ax.hexbin(y, pred_best, gridsize=80, cmap='viridis', bins='log', mincnt=1)
lim = [0, max(y.max(), pred_best.max())]
ax.plot(lim, lim, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set(title=f'{best_name}: Actual vs. Predicted price', xlabel='Actual price (₹)', ylabel='Predicted price (₹)')
ax.xaxis.set_major_formatter(rupee); ax.yaxis.set_major_formatter(rupee)
ax.legend(); fig.colorbar(hb, ax=ax, label='log10(count)')

ax = axes[1]
sns.histplot(resid, bins=120, ax=ax, color=MODEL_COLORS[best_name])
ax.axvline(0, color='red', linestyle='--')
ax.set(title=f'{best_name}: Residuals (actual − predicted)', xlabel='Error (₹)', ylabel='Count')
ax.set_xlim(np.percentile(resid, 0.5), np.percentile(resid, 99.5))

plt.tight_layout(); plt.show()

In [ ]:
# Error broken down by class: business fares are much larger, so their absolute errors are larger too
err = pd.DataFrame({'class': X['class'], 'abs_err': np.abs(resid), 'price': y})
err_by_class = err.groupby('class').agg(rows=('abs_err', 'size'), MAE=('abs_err', 'mean'), avg_price=('price', 'mean'))
err_by_class['MAE as % of avg price'] = 100 * err_by_class['MAE'] / err_by_class['avg_price']
err_by_class.style.format({'rows': '{:,}', 'MAE': '₹{:,.0f}', 'avg_price': '₹{:,.0f}', 'MAE as % of avg price': '{:.1f}%'})

## 13. Train Final Model on the Full Dataset & Save It
Cross-validation was used to **choose** the model. The chosen model is now retrained on **100% of the data** and saved with `joblib`.

The deployed file must be under 100 MB (GitHub's limit). Large forest models like Extra Trees can produce files of hundreds of MB,
so the code tries models in rank order and deploys the **best-ranked model that fits under the limit**.

In [ ]:
MODEL_FILE = 'flight_price_model.joblib'
deployed_name = None
for name in order:
    print(f'Training {name} on all {len(X):,} rows...', end=' ')
    t0 = time.perf_counter()
    final_pipe = make_pipeline(models[name]).fit(X, y)
    fit_s = time.perf_counter() - t0
    joblib.dump(final_pipe, MODEL_FILE, compress=3)
    size_mb = os.path.getsize(MODEL_FILE) / 1e6
    print(f'done in {fit_s:,.1f}s, file size {size_mb:,.1f} MB')
    if size_mb <= MAX_MODEL_MB:
        deployed_name = name
        break
    print(f'   ↳ too large for GitHub (> {MAX_MODEL_MB} MB), trying the next-best model')

rank = int(summary.loc[deployed_name, 'Rank'])
print(f'\n✅ Deployed model: {deployed_name} (CV rank #{rank}), saved as {MODEL_FILE} ({size_mb:,.1f} MB)')
if deployed_name != best_name:
    gap = summary.loc[deployed_name, 'RMSE_mean'] - summary.loc[best_name, 'RMSE_mean']
    print(f'   ({best_name} was most accurate by ₹{gap:,.0f} RMSE but its file is too large to deploy for free.)')

In [ ]:
# Feature importance of the deployed model (tree models only)
est = final_pipe.named_steps['model']
names = final_pipe.named_steps['prep'].get_feature_names_out()
names = [n.split('__', 1)[1] for n in names]
if hasattr(est, 'feature_importances_'):
    imp = pd.Series(est.feature_importances_, index=names).sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(imp.index[::-1], imp.values[::-1], color=MODEL_COLORS[deployed_name])
    ax.set(title=f'Top 15 features: {deployed_name}', xlabel='Importance')
    plt.tight_layout(); plt.show()
else:
    coef = pd.Series(est.coef_, index=names).sort_values(key=np.abs, ascending=False).head(15)
    coef[::-1].plot.barh(figsize=(9, 6), title=f'Top 15 coefficients: {deployed_name}'); plt.show()

In [ ]:
# Sanity check: reload the saved file and predict one ticket exactly as the web app will
reloaded = joblib.load(MODEL_FILE)
example = pd.DataFrame([{
    'stops': 'one', 'class': 'Economy', 'airline': 'Vistara', 'source_city': 'Delhi',
    'departure_time': 'Morning', 'arrival_time': 'Evening', 'destination_city': 'Mumbai',
    'duration': 10.0, 'days_left': 15,
}])[FEATURES]
print(f'Example prediction (Delhi → Mumbai, Vistara, Economy, 1 stop, 15 days ahead): ₹{reloaded.predict(example)[0]:,.0f}')

In [ ]:
# Metadata for the web app: dropdown options, CV results and library versions
meta = {
    'model_name': deployed_name,
    'best_cv_model': best_name,
    'trained_on_rows': int(len(X)),
    'trained_at': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'n_folds': N_FOLDS,
    'features': FEATURES,
    'options': {c: sorted(df[c].unique().tolist()) for c in NOMINAL_COLS},
    'stops_options': ORDINAL_CATS[0],
    'class_options': ORDINAL_CATS[1],
    'airline_classes': {a: sorted(g['class'].unique().tolist()) for a, g in df.groupby('airline')},
    'duration_range': [float(df.duration.min()), float(df.duration.max()), float(df.duration.median())],
    'days_left_range': [int(df.days_left.min()), int(df.days_left.max())],
    'cv_results': [
        {'model': m, 'rank': int(r.Rank), 'rmse': float(r.RMSE_mean), 'rmse_std': float(r.RMSE_std),
         'mae': float(r.MAE_mean), 'r2': float(r.R2_mean), 'train_s': float(r.Train_total_s)}
        for m, r in summary.iterrows()],
    'versions': {'python': platform.python_version(), 'scikit-learn': sklearn.__version__,
                 'numpy': np.__version__, 'pandas': pd.__version__},
}
with open('model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

# requirements.txt pinned to the SAME versions used here, otherwise the model may fail to load on the server
reqs = ['streamlit', f'scikit-learn=={sklearn.__version__}', f'numpy=={np.__version__}',
        f'pandas=={pd.__version__}', f'joblib=={joblib.__version__}']
if deployed_name == 'XGBoost':
    reqs.append(f'xgboost-cpu=={xgboost.__version__}')   # CPU-only build: same library, much smaller download
if deployed_name == 'LightGBM':
    reqs.append(f'lightgbm=={lightgbm.__version__}')
with open('requirements.txt', 'w') as f:
    f.write('\n'.join(reqs) + '\n')
print(open('requirements.txt').read())
print('These versions install on Streamlit Cloud with Python 3.12, 3.13 or 3.14.')

In [ ]:
# Save the results tables too (useful for the report) and download everything
fold_df.to_csv('cv_fold_results.csv', index=False)
summary.to_csv('cv_summary.csv')

import zipfile
bundle = 'deployment_files.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for fn in [MODEL_FILE, 'model_meta.json', 'requirements.txt', 'cv_fold_results.csv', 'cv_summary.csv']:
        z.write(fn)
print(f'Created {bundle} ({os.path.getsize(bundle)/1e6:,.1f} MB)')

try:
    from google.colab import files
    files.download(bundle)
except ImportError:
    print('Not running on Colab: files are in the current folder.')

## 14. Deployment (Streamlit Community Cloud)

The trained model is served by a **Streamlit web app** (`app.py`) hosted for free on **Streamlit Community Cloud**,
the same platform used for the instructor's Titanic demo.

**Architecture:** user fills in flight details in the browser → `app.py` builds a one-row DataFrame → the saved
`Pipeline` (preprocessing + model) predicts the price → the app shows the estimated fare.

**Steps followed:**
1. Unzipped `deployment_files.zip` from the previous cell.
2. Created the public GitHub repo https://github.com/rameenejaz/flight-prediction-model containing `app.py`, `flight_price_model.joblib`,
   `model_meta.json`, `requirements.txt` and `.streamlit/config.toml` (theme).
3. On **share.streamlit.io** → *Create app* → selected the repo, branch `main`, main file `app.py`, and deployed.

**Version compatibility:** the first deployment failed because the server runs a newer Python (3.14) than Colab, and the
library versions used for training had no installers for it. The fix was to pin library versions that support Python
3.12 – 3.14 (setup cell in Section 4) in both this notebook and `requirements.txt`, so the model trained here loads identically on the server.

**Live app:** https://flight-prediction-model-24jznugnvxzgr5mksxuxwf.streamlit.app

## 15. Conclusion

In [ ]:
# Auto-generated summary of YOUR results (use it to write the conclusion below in your own words)
fastest = summary['Train_total_s'].idxmin()
print(f'Dataset: {len(X):,} rows, {N_FOLDS}-fold CV, 5 algorithms, total run time {total_minutes:.1f} min\n')
for m, r in summary.iterrows():
    print(f'#{int(r.Rank)} {m:<17} RMSE ₹{r.RMSE_mean:>8,.0f} ± {r.RMSE_std:>5,.0f} | R² {r.R2_mean:.4f} | '
          f'train {r.Train_total_s:>8,.1f}s ({r["Speed vs fastest"]:,.0f}× fastest)')
print(f'\nMost accurate: {best_name}.  Fastest: {fastest}.  Deployed: {deployed_name}.')

### Summary of results (10-fold CV, 300,153 flights)

| Rank | Model | Mean RMSE (₹) | MAE (₹) | R² |
|---|---|---|---|---|
| 1 | **XGBoost** | **2,599 ± 27** | 1,385 | **0.9869** |
| 2 | LightGBM | 2,633 ± 32 | 1,409 | 0.9865 |
| 3 | Extra Trees | 2,664 ± 37 | 1,210 | 0.9862 |
| 4 | AdaBoost | 4,430 ± 49 | 2,682 | 0.9619 |
| 5 | Ridge Regression | 6,811 ± 43 | 4,519 | 0.9100 |

**1. Best model.** XGBoost achieved the lowest RMSE (about ₹2,600) and the highest R² (0.987), so it explains almost 99% of the
variation in ticket prices. LightGBM and Extra Trees were very close behind: the top three models are within about 2.5% of each other.

**2. Linear vs. tree-based models.** Ridge Regression was clearly the weakest (RMSE about ₹6,800, R² 0.91), roughly 2.6 times the error of XGBoost.
Flight prices depend on *interactions* between features: for example, a business-class ticket booked a few days before departure
costs far more than the separate effects of "business class" and "few days left" would suggest. A linear model adds feature effects
together and cannot capture this, while decision trees split on combinations of features naturally.

**3. Boosting methods.** Modern gradient boosting (XGBoost, LightGBM) clearly beat classic AdaBoost (RMSE about ₹4,430). AdaBoost only re-weights
the training samples between trees, whereas gradient boosting fits each new tree directly to the remaining errors and adds
regularisation, which works much better on this data.

**4. RMSE vs. MAE.** Extra Trees had the *lowest MAE* (about ₹1,210) but a higher RMSE than XGBoost. This means it is more accurate on typical
tickets but makes a few larger mistakes (RMSE punishes large errors more heavily). These large errors are most likely on expensive business-class
fares, where absolute errors are naturally bigger.

**5. Accuracy vs. computational cost.** Ridge trained in seconds but was inaccurate. Among the accurate models, LightGBM was the most efficient:
roughly 2–3 times faster than XGBoost for almost the same accuracy, while Extra Trees and AdaBoost were the slowest (several minutes each for
10 folds). If training time mattered more than the last 1% of accuracy, LightGBM would be the best choice.

**6. Stability.** All models had small standard deviations across the 10 folds (₹27–49 RMSE), so the results do not depend on how the
data was split. XGBoost had the smallest spread, making it the most stable as well as the most accurate.

**7. Deployed model.** XGBoost was deployed because it had the best accuracy and stability, its model file is small (about 2.3 MB, well under
GitHub's 100 MB limit), and it predicts a single ticket in milliseconds, which suits a web app.

**8. Limitations and future work.** No hyper-parameter tuning was done (all models used fixed, reasonable settings), so tuning with
`RandomizedSearchCV` could improve every model. The data covers only six Indian cities and a short booking period, so predictions may not
apply to other routes or future seasons, and real airline prices also depend on factors not in the dataset (demand, holidays, fuel prices).